<h1 style="text-align: center;"> Project name </h1>

<div style="display: flex; justify-content: space-around;">

<div style="width: 30%; text-align: center;">
<strong>Jie Zhao</strong>  
<br>
jiz273@g.harvard.edu
</div>

<div style="text-align: center; width: 80%; margin: 0 auto;">
    <strong>Abstract</strong><br>
    FER-2013 is a dataset of facial expressions that contains 35,887 grayscale images of faces with seven different emotions: anger, disgust, fear, happiness, sadness, surprise, and neutral. In this project, we explore various different approaches to classifying emotions, custom convolutional neural networks (CNNs), transfer learning and vision transformers. Prior work on this dataset without an auxiliary dataset ranges from 70-75% accuracy, where human classification accuracy is benchmarked between 60 and 70%. Most relied heavily on CNNs but in this project we also tested vision transformers. We use an autoencoder to filter out outlying images based on reconstruction data. Using this filtered dataset, our custom U-Net model with squeeze-excite blocks achieves a test accuracy of 64%, matching human benchmarks. Our most performant model is transfer learning with a ResNet50 model, which achieves a test accuracy of 68.5%, comparable to other papers. Other papers have found higher accuracies in the range of 73-78% but these were done with an auxiliary dataset or with a multi-label setup. Although novel, the vision transformer did not perform well, only marginally better than random. This is most likely due to the small sample size, since models with weaker inductive biases need more data to learn.
</div>

## Table of Contents

**1. [Introduction](#introduction)**  
 1.1 [Problem Statement](#problem-statement)  

**2. [Installation, Configuration and Set UP](#introduction)**  

**3. [Dataset and Data Preparation](#comprehensive-eda-review)**  
 2.1 [Data Description](#data-description)  
 2.2 [Explortaty Analysis]
 2.3 [Data Cleaning]
 2.4 [Data Analysis](#understand)  

**4. [Step-by-Step Project Development](#research-question)**  

**5. [Results and Demonstration](#baseline-model)**  

**6. [Uses and Benefits](#final-model)**  

**7. [Challenges / Lesson Learnt]

**8. [Youtube Video Links] 

**9. [References] 

**10. [Appendix] 



 ## 1. Introduction

### 1.1 Business Context

Food establishments in Boston are inspected to protect public health and ensure compliance with food safety regulations. These inspections generate large amounts of public data, including violation descriptions, inspection results, dates, locations, and business information.

However, this data is difficult for non-technical users to explore. Restaurant owners, city analysts, public health teams and general publics may not easily know:

which violations are increasing
which neighborhoods show higher inspection risk
which establishments have repeated issues
which violation types are most common
which past inspections are similar to a current problem

### 1.2 Problem Statement

## 2. Installation, Configuration and Set UP

2.1 Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pyarrow as pa




In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv

# LangChain components for  RAG system
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain.schema import Document


# Load your environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("All libraries loaded successfully!")

ImportError: cannot import name '_V2StreamingCallbackHandler' from 'langchain_core.tracers._streaming' (/Users/jiezhao/miniconda3/envs/llm_langchain/lib/python3.14/site-packages/langchain_core/tracers/_streaming.py)

## 3. Dataset and Data Preparation

The Bostn Health Division of the Department of Inspectional Services ensures that all food establishments in the City of Boston meet relevant sanitary codes and standards. Businesses that serve food are inspected at least once a year, and follow-up inspections are performed on high risk establishments. Health inspections are also conducted in response to complaints of unsanitary conditions or illness. 

The Boston Food Establishment Inspections dataset contains the outcomes of food establishment inspections conducted in the city’s greater area since 2006. Updated daily, this dataset provides details about individual inspections and results of businesses serving food. For this analysis, we are working with a static version of the dataset, comprising 27 columns, and 884608 individual records.

The dataset includes information such as the time and location of each inspection, the business entity responsible, and the licensing details. It also records inspection outcomes, violations noted, and any follow-up actions or comments, offering a comprehensive view of Boston’s food safety practices.

3.1 Load data and inspect

In [147]:
#Load and inspect data
df = pd.read_csv('data/food_inspection_report_raw.csv')
df.info()

/var/folders/0x/_5wx0swx0vxgsyrkv847t75w0000gn/T/ipykernel_22344/3106629136.py:2: DtypeWarning: Columns (0: zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/food_inspection_report_raw.csv')


<class 'pandas.DataFrame'>
RangeIndex: 884608 entries, 0 to 884607
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   businessname  884608 non-null  str    
 1   dbaname       8431 non-null    str    
 2   legalowner    575201 non-null  str    
 3   namelast      884608 non-null  str    
 4   namefirst     508539 non-null  str    
 5   licenseno     884608 non-null  int64  
 6   issdttm       883712 non-null  str    
 7   expdttm       883928 non-null  str    
 8   licstatus     884608 non-null  str    
 9   licensecat    884608 non-null  str    
 10  descript      884608 non-null  str    
 11  result        884608 non-null  str    
 12  resultdttm    878210 non-null  str    
 13  violation     823807 non-null  str    
 14  viol_level    823807 non-null  str    
 15  violdesc      816937 non-null  str    
 16  violdttm      823804 non-null  str    
 17  viol_status   823807 non-null  str    
 18  status_date   3

In [3]:
#understand the data structure
print('size of data', df.shape)
print('columns', df.columns)
print('describe of data', df.describe())


size of data (884608, 26)
columns Index(['businessname', 'dbaname', 'legalowner', 'namelast', 'namefirst',
       'licenseno', 'issdttm', 'expdttm', 'licstatus', 'licensecat',
       'descript', 'result', 'resultdttm', 'violation', 'viol_level',
       'violdesc', 'violdttm', 'viol_status', 'status_date', 'comments',
       'address', 'city', 'state', 'zip', 'property_id', 'location'],
      dtype='str')
describe of data            licenseno    property_id
count  884608.000000  727234.000000
mean   107065.636450  150178.259790
std    144712.285047  103322.491624
min        54.000000       0.000000
25%     22153.000000   77703.000000
50%     28531.000000  155991.000000
75%    125523.000000  157956.000000
max    624593.000000  460598.000000


Explore potential meaningful columns for our analysis deeper to understand the type and cardinality

In [4]:
# violation and description columns
print('violcation code carinality\n\n', df['violation'].value_counts())
print('='*100)
print('violcation description carinality\n\n',df['violdesc'].value_counts())

violcation code carinality

 violation
23-4-602.13            43973
37-6-501.11-.12        39951
15-4-202.16            35183
36-6-501.11-.12        33806
08-3-305-307.11        30211
                       ...  
590.004/4-204.123-C        1
02-3-305.11(2)             1
590.003/3-801.11-C         1
                           1
590.005/5-402.14-PF        1
Name: count, Length: 462, dtype: int64
violcation description carinality

 violdesc
Non-Food Contact Surfaces Clean                                                                   43973
Improper Maintenance of Walls/Ceilings                                                            39951
Non-Food Contact Surfaces                                                                         35183
Improper Maintenance of Floors                                                                    33806
Food Protection                                                                                   30211
                                      

In [5]:
# more star means more severe
df['viol_level'].value_counts()

viol_level
*       579434
***     126544
**      110958
-         6869
1919         1
             1
Name: count, dtype: int64

| Raw value | Interpreted meaning   | Action                    |
| --------- | --------------------- | ------------------------- |
| `*`       | Low severity          | map to severity_score = 1 |
| `**`      | Medium severity       | map to severity_score = 2 |
| `***`     | High severity         | map to severity_score = 3 |
| `-`       | Missing / unspecified | null                      |
| `1919`    | malformed data        | remove/null               |


In [6]:
# require mapping violation level from stars to understanable severity, and assign numerical scores to severity
severity_map = {
    "*": "low",
    "**": "medium",
    "***": "high",
    "-": None,
    "1919": None
}
severity_score_map= {
    "*": 1,
    "**": 2,
    "***": 3
}


In [7]:
df['viol_status'].value_counts()

viol_status
Fail    452084
Pass    365983
          5740
Name: count, dtype: int64

In [38]:
df[df['result'] =='Pass']

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,descript,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
2385,2nd Cup Cafe,NaN,AND DEVELOPMENT LLC,Tea Time Inc.,NaN,26681,2011-03-04 17:41:12+00,2010-12-31 05:00:00+00,Inactive,FS,Eating & Drinking,Pass,2007-09-21 04:00:00+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101 BRIGHTON AV,ALLSTON,MA,02134,20067.0,"(42.35308144420537, -71.13062406224189)"
3502,68 Chinese Fast Food,NaN,P & C COMPANY: TONG'S FAS T FOOD,Tony Q Huynh,NaN,20591,2012-03-01 15:29:40+00,2016-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,02-3-602.11-.12/3-302.12,*,Food Container Labels,2007-06-13 18:52:00+00,Pass,2007-06-15 19:27:28+00,NaN,48 WINTER ST,BOSTON,MA,02111,155968.0,"(42.355863979184754, -71.06189896417312)"
3503,68 Chinese Fast Food,NaN,P & C COMPANY: TONG'S FAS T FOOD,Tony Q Huynh,NaN,20591,2012-03-01 15:29:40+00,2016-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,15-4-202.16,*,Non-Food Contact Surfaces,2007-06-13 18:52:00+00,Pass,2007-06-15 19:27:18+00,NaN,48 WINTER ST,BOSTON,MA,02111,155968.0,"(42.355863979184754, -71.06189896417312)"
3504,68 Chinese Fast Food,NaN,P & C COMPANY: TONG'S FAS T FOOD,Tony Q Huynh,NaN,20591,2012-03-01 15:29:40+00,2016-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,23-4-602.13,*,Non-Food Contact Surfaces Clean,2007-06-13 18:53:00+00,Pass,2007-06-15 19:27:03+00,NaN,48 WINTER ST,BOSTON,MA,02111,155968.0,"(42.355863979184754, -71.06189896417312)"
3505,68 Chinese Fast Food,NaN,P & C COMPANY: TONG'S FAS T FOOD,Tony Q Huynh,NaN,20591,2012-03-01 15:29:40+00,2016-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,36-6-501.11-.12,*,Improper Maintenance of Floors,2007-06-13 18:56:00+00,Pass,2007-06-15 19:26:53+00,NaN,48 WINTER ST,BOSTON,MA,02111,155968.0,"(42.355863979184754, -71.06189896417312)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
879071,YUMMY YUMMY,NaN,DIPASQUALE ALFRED P,MING CHEN,AKA PEKING WOK,21312,2007-11-21 13:51:22+00,2008-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,23-4-602.13,*,Non-Food Contact Surfaces Clean,2007-05-29 18:18:00+00,Pass,2007-06-08 19:12:39+00,NaN,2360 WASHINGTON ST,ROXBURY,MA,2119.0,144493.0,"(42.32937000050038, -71.0844600020372)"
879072,YUMMY YUMMY,NaN,DIPASQUALE ALFRED P,MING CHEN,AKA PEKING WOK,21312,2007-11-21 13:51:22+00,2008-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,36-6-501.11-.12,*,Improper Maintenance of Floors,2007-05-29 18:22:00+00,Pass,2007-06-08 19:12:01+00,NaN,2360 WASHINGTON ST,ROXBURY,MA,2119.0,144493.0,"(42.32937000050038, -71.0844600020372)"
879073,YUMMY YUMMY,NaN,DIPASQUALE ALFRED P,MING CHEN,AKA PEKING WOK,21312,2007-11-21 13:51:22+00,2008-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,37-6-201.11,*,Walls/Ceilings Designed Constructed Installed,2007-05-29 18:23:00+00,Pass,2007-06-08 19:11:48+00,NaN,2360 WASHINGTON ST,ROXBURY,MA,2119.0,144493.0,"(42.32937000050038, -71.0844600020372)"
879074,YUMMY YUMMY,NaN,DIPASQUALE ALFRED P,MING CHEN,AKA PEKING WOK,21312,2007-11-21 13:51:22+00,2008-01-01 04:59:00+00,Inactive,FT,Eating & Drinking w/ Take Out,Pass,NaN,38-6-202.11,*,Fixture's not properly shielded,2007-05-29 18:21:00+00,Pass,2007-06-08 19:12:18+00,NaN,2360 WASHINGTON ST,ROXBURY,MA,2119.0,144493.0,"(42.32937000050038, -71.0844600020372)"


In [8]:
df['result'].value_counts()

result
HE_Fail       369707
HE_Pass       282095
HE_Filed       92851
HE_FailExt     73291
HE_Hearing     27393
HE_NotReq      23985
HE_TSOP         7669
HE_VolClos      2839
HE_OutBus       2521
Pass             964
HE_Closure       711
Fail             238
HE_FAILNOR       146
HE_Misc          128
DATAERR           42
HE_Hold           17
Failed             6
Closed             2
PassViol           2
NoViol             1
Name: count, dtype: int64

In [9]:
df[df['result']=='PassViol']

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,...,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
492580,MELO'S GROCERY,NaN,ESPAILLAT MANUEL A,GUZMAN,ORQUIDIA,24704,2009-04-15 18:29:55+00,2008-01-01 04:59:00+00,Inactive,RF,...,NaN,NaN,NaN,NaN,331 CENTRE ST,JAMAICA PLAIN,MA,2130,27981.0,"(42.32287000024958, -71.10500000140604)"
753673,Taj Boston,NaN,MPE HOTEL I LLC,ALBRIGHT,MAUREEN,24892,2012-01-11 13:47:28+00,2017-01-01 04:59:00+00,Inactive,RF,...,NaN,NaN,NaN,NaN,15 ARLINGTON ST,BOSTON,MA,2116.0,4827.0,"(42.35282999954591, -71.07160000160395)"


Each record corresponds to a single violation, meaning that multiple records can exist for the same inspection if several violations are found during the procedure. Each violation is documented with its own description and severity classification, but records from the same inspection share the inspection result and date-time fields.

In [10]:
pd.set_option('display.max_columns', None) 

In [11]:
df.head(15)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,descript,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clea...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Fail,2018-03-20 14:54:25+00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,NaN,Several dented cans found on storage shelves. ...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2018-08-08 15:54:00+00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,NaN,Wet wiping cloths found on counter tops . Remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
5,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2019-02-04 18:06:00+00,02-3-602.11-.12/3-302.12,*,Food Container Labels,2019-02-04 18:06:00+00,Fail,NaN,No labels on bulk containers . Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
6,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Filed,2019-02-04 18:06:00+00,10-3-304.12,*,Food Utensil Storage,2019-02-04 18:06:00+00,Fail,NaN,Scoops found submerges in flour and other bulk...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
7,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_NotReq,2022-03-15 19:22:42.093+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
8,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Pass,2017-08-11 14:10:25+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
9,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,Eating & Drinking,HE_Pass,2017-12-15 18:58:58+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


Explore comments column, we can see the comments usually include the observation of violation, and improvement suggestion.

In [12]:
pd.set_option('display.max_colwidth', None)
df['comments'][:8]

0                                                                               One staff person without hair restraint. Provide
1                                                                     Caked on food debris on can opener blade. Clean to remove.
2                      Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.
3    Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.
4                                      Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.
5                                                                                         No labels on bulk containers . Provide
6                                      Scoops found submerges in flour and other bulk container. Store properly with handles up.
7                                                                                                

In [148]:
#description of inspection result
ref = pd.read_csv('data/InspectionResult_description.csv')
# Standardize column names
ref.columns = ref.columns.str.lower().str.strip()
ref

,inspectionresult,inferreddescription
0,HE_Fail,Inspection failed; violations found
1,HE_Pass,Inspection passed; no issues found
2,HE_Filed,Minor violations found; no urgent follow-up required
3,HE_FailExt,Extended failure from prior inspection
4,HE_Hearing,Violations being addressed; follow-up required
5,HE_NotReq,Inspection not required; no immediate need
6,HE_TSOP,Temporary suspension of permit issued
7,HE_OutBus,Business closed; out of operation
8,HE_VolClos,Voluntary closure by business to avoid penalties
9,HE_Closure,Forced closure due to critical violations


We aggregate inspection results into category and assign risk scores

In [14]:
result_group_map = {
    "HE_Pass": "pass",
    "Pass": "pass",
    
    "HE_Filed": "minor_violation",

    "HE_Fail": "fail",
    "Fail": "fail",
    "Failed": "fail",
    "HE_FAILNOR": "fail",

    "HE_FailExt": "extended_fail",
    "HE_Hearing": "hearing",

    "HE_TSOP": "temporary_suspension",
    "HE_VolClos": "voluntary_closure_avoid",
    "HE_Closure": "forced_closure",

    "HE_OutBus": "out_of_business",
    "Closed": "closed",

    "HE_NotReq": "not_required",
    "HE_Misc": "misc",
    "DATAERR": "data_error",
    "HE_Hold": "hold"
}

risk_score_map = {
    "pass": 1,
    "minor_violation": 2,
    "hold": 2,

    "fail": 3,
    "extended_fail": 3,
    "hearing": 3,

    "temporary_suspension": 4,
    "voluntary_closure_avoid": 4,
    "forced_closure": 4,

    "out_of_business": None,
    "closed": None,
    "not_required": None,
    "misc": None,
    "data_error": None,
}



#upon initial inspection of the data, we decided the columns below are relative and meaningful for our anlaysis.

- businessname: business name
- licenseno: Key identifier for the business/ restaurant
-result: result of  inspection
-resultdttm: date on which the results were generated
-violation: coding of law regulation related to violations
- viol_level: level of violation
-violdesc: reason of violation
- violdttm: date on which violation status was generated
- viol_status: status for violation: Fail or Pass
-status_date: date on which violation status was set to pass
- comments: comments given to the food establishment for improvement
- address: address of the business
- zipcode: zipcode of the business
- location: latitude and longitude of the business location

In [149]:
cols = [
    "businessname",
    "licenseno",
    "result",
    "resultdttm",
    "violation",
    "viol_level",
    "violdesc",
    "violdttm",
    "viol_status",
    "comments",
    "address",
    "zip",
    "location",
]

data = df[cols].copy()
data.head()

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)"


In [150]:
data["severity_level"] = data["viol_level"].map(severity_map)
data["severity_score"] = data["viol_level"].map(severity_score_map)

data["result_group"] = data["result"].map(result_group_map)
data["risk_score"] = data["result_group"].map(risk_score_map)

In [151]:
# merge the description of inspection result to the main dataframe
data = data.merge(
    ref[["inspectionresult", "inferreddescription"]],
    left_on="result",
    right_on="inspectionresult",
    how="left"
)
data = data.drop(columns=["inspectionresult"])

In [152]:
# the analysis focuses on violation related content, we drop the rows with missing violation, result, viol_level, viol_status
data.dropna(subset=['violation','result','viol_level','viol_status'], inplace=True)


823,807 usable rows,1-2% percent of the data is missing.

In [153]:
data.info()

<class 'pandas.DataFrame'>
Index: 823807 entries, 0 to 884607
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   businessname         823807 non-null  str    
 1   licenseno            823807 non-null  int64  
 2   result               823807 non-null  str    
 3   resultdttm           820159 non-null  str    
 4   violation            823807 non-null  str    
 5   viol_level           823807 non-null  str    
 6   violdesc             816937 non-null  str    
 7   violdttm             823804 non-null  str    
 8   viol_status          823807 non-null  str    
 9   comments             797312 non-null  str    
 10  address              823678 non-null  str    
 11  zip                  823322 non-null  object 
 12  location             753462 non-null  str    
 13  severity_level       816936 non-null  str    
 14  severity_score       816936 non-null  float64
 15  result_group         823807 non-n

In [154]:
# handling na
# need date time data for analysis 
data["resultdttm"] = pd.to_datetime(df["resultdttm"], errors="coerce")
data["resultdttm"] = pd.to_datetime(data["resultdttm"], errors="coerce")
data["year"] = data["resultdttm"].dt.year
data["month"] = data["resultdttm"].dt.strftime("%Y-%m")
data = data.dropna(subset=["resultdttm"])

#remove most current month 2026-05 as it's not completed month
data = data[data['month'] != '2026-05']
# fill na with unknown
data["violdesc"] = data["violdesc"].fillna("Unknown")
data["comments"] = data["comments"].fillna("No comment")
data["address"] = data["address"].fillna("Unknown")
data["inferreddescription"] = data["inferreddescription"].fillna("Unknown")
data["address"] = data["address"].fillna("Unknown")
data["zip"] = data["zip"].astype("string")


In [155]:
data.info()

<class 'pandas.DataFrame'>
Index: 819596 entries, 0 to 884607
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   businessname         819596 non-null  str                
 1   licenseno            819596 non-null  int64              
 2   result               819596 non-null  str                
 3   resultdttm           819596 non-null  datetime64[us, UTC]
 4   violation            819596 non-null  str                
 5   viol_level           819596 non-null  str                
 6   violdesc             819596 non-null  str                
 7   violdttm             819594 non-null  str                
 8   viol_status          819596 non-null  str                
 9   comments             819596 non-null  str                
 10  address              819596 non-null  str                
 11  zip                  819122 non-null  string             
 12  location          

In [22]:
data.head(2)

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription,year,month
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",medium,2.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03


In [37]:
data.head(10)

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription,year,month
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",medium,2.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03
2,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,M-2-103.11,***,PIC Performing Duties,2018-03-20 14:54:25+00,Fail,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",high,3.0,fail,3.0,Inspection failed; violations found,2018.0,2018-03
3,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00:00,08-3-305-307.11,*,Food Protection,2018-08-08 15:54:00+00,Fail,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,minor_violation,2.0,Minor violations found; no urgent follow-up required,2018.0,2018-08
4,1000 Degrees Pizza,313440,HE_Filed,2018-08-08 15:54:00+00:00,21-3-304.14,*,Wiping Cloths Clean Sanitize,2018-08-08 15:54:00+00,Fail,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,minor_violation,2.0,Minor violations found; no urgent follow-up required,2018.0,2018-08
5,1000 Degrees Pizza,313440,HE_Filed,2019-02-04 18:06:00+00:00,02-3-602.11-.12/3-302.12,*,Food Container Labels,2019-02-04 18:06:00+00,Fail,No labels on bulk containers . Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,minor_violation,2.0,Minor violations found; no urgent follow-up required,2019.0,2019-02
6,1000 Degrees Pizza,313440,HE_Filed,2019-02-04 18:06:00+00:00,10-3-304.12,*,Food Utensil Storage,2019-02-04 18:06:00+00,Fail,Scoops found submerges in flour and other bulk container. Store properly with handles up.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,minor_violation,2.0,Minor violations found; no urgent follow-up required,2019.0,2019-02
10,1000 Degrees Pizza,313440,HE_Pass,2018-03-23 15:46:18+00:00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-23 15:46:18+00,Pass,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,pass,1.0,Inspection passed; no issues found,2018.0,2018-03
11,1000 Degrees Pizza,313440,HE_Pass,2018-03-23 15:46:18+00:00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-23 15:46:18+00,Pass,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",medium,2.0,pass,1.0,Inspection passed; no issues found,2018.0,2018-03
12,1000 Degrees Pizza,313440,HE_Pass,2018-03-23 15:46:18+00:00,M-2-103.11,***,PIC Performing Duties,2018-03-23 15:46:18+00,Pass,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",high,3.0,pass,1.0,Inspection passed; no issues found,2018.0,2018-03


In [156]:
#write it to parquet file, more efficient for database use

data.to_parquet("data/food_inspections_clean.parquet", engine="pyarrow",index=False)

3. DuckDB setup

Load cleaned data into DuckDB

In [56]:
data.columns

Index(['businessname', 'licenseno', 'result', 'resultdttm', 'violation',
       'viol_level', 'violdesc', 'violdttm', 'viol_status', 'comments',
       'address', 'zip', 'location', 'severity_level', 'severity_score',
       'result_group', 'risk_score', 'inferreddescription_x',
       'inferreddescription_y', 'year', 'month'],
      dtype='str')

In [34]:
import duckdb

con = duckdb.connect("data/food_inspections_clean.duckdb")

con.execute("""
CREATE OR REPLACE TABLE inspections AS
SELECT *
FROM read_parquet('data/food_inspections_clean.parquet')
""")

con.execute("SELECT COUNT(*) FROM inspections").fetchall()

[(820092,)]

4. SQL functions 

In [104]:
# SQL functions 1 - top violation types,What violations are most common?
def top_violation_types(year=2025, severity_level=None, limit=15):
    query = """
    SELECT 
        violdesc,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE violdesc IS NOT NULL 
      AND violdesc <> 'Unknown violation'
      AND severity_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
      AND (? IS NULL OR severity_level = ?)
    GROUP BY violdesc
    ORDER BY violation_count DESC
    LIMIT ?
    """
    return con.execute(
        query,
        [year, year, severity_level, severity_level, limit]
    ).df()

Business question: Are food safety violations increasing or decreasing over time? It helps identify trend, seanonality, pattern 

In [59]:
# SQL functions 2 - violation trend by month
def violation_trend_by_month(start_year=2021):
    query = """
    SELECT
        month,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE month IS NOT NULL
      AND violdesc IS NOT NULL
      AND violdesc <> 'Unknown violation'
      AND severity_score IS NOT NULL
      AND year >= ?
    GROUP BY month
    ORDER BY month
    """
    return con.execute(query, [start_year]).df()

Business question: Are high-severity violations increasing faster than mild violations? Distinguishes whether risk is worsening or whether only minor violations are increasing.

In [60]:
def violation_trend_by_month_and_severity(start_year=2021):
    query = """
    SELECT
        month,
        severity_level,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE month IS NOT NULL
      AND violdesc IS NOT NULL
      AND violdesc <> 'Unknown violation'
      AND severity_score IS NOT NULL
      AND severity_level IS NOT NULL
      AND year >= ?
    GROUP BY month, severity_level
    ORDER BY month, severity_level
    """
    return con.execute(query, [start_year]).df()

In [ ]:
def severity_by_zip(limit=15):
    query = """
    SELECT
        zip,
        COUNT(*) AS total_violations,
        AVG(severity_score) AS avg_severity,
        SUM(CASE WHEN severity_level = 'high' THEN 1 ELSE 0 END) AS high_severity_count
    FROM inspections
    WHERE zip IS NOT NULL
      AND severity_score IS NOT NULL
    GROUP BY zip
    ORDER BY high_severity_count DESC, avg_severity DESC
    LIMIT ?
    """
    return con.execute(query, [limit]).df()

Business question: Which restaurants repeatedly fail inspections or have consistently severe violations?

In [ ]:
def top_violation_owner(year=2025, limit=15):
    query = """
    SELECT
        businessname,
        address,
        COUNT(*) AS total_violations,
        SUM(CASE WHEN result_group IN (
            'fail',
            'extended_fail',
            'hearing',
            'temporary_suspension',
            'voluntary_closure_avoid',
            'forced_closure'
        ) THEN 1 ELSE 0 END) AS serious_outcome_count,
        AVG(severity_score) AS avg_severity
    FROM inspections
    WHERE businessname IS NOT NULL
      AND severity_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
    GROUP BY businessname, address
    HAVING COUNT(*) > 10
    ORDER BY serious_outcome_count DESC, avg_severity DESC
    LIMIT ?
    """
    return con.execute(query, [year, year, limit]).df()

In [171]:
data[data["businessname"].str.contains("mala town", case=False, na=False)]

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription,year,month
469216,Mala Town,571486,HE_Fail,2024-09-09 15:37:54+00:00,590.003/3-306.11-P,***,Food Display-Preventing Contamination by Consumers (P),2024-09-09 15:37:54+00,Fail,Establishment has ordered and is waiting for sneeze guards for the self service refrigeration- Install when recieved,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",high,3.0,fail,3.0,Inspection failed; violations found,2024.0,2024-09
469217,Mala Town,571486,HE_Fail,2024-09-09 15:37:54+00:00,590.005/5-303.12-C,*,Protective Cover or Device (C),2024-09-09 15:37:54+00,Fail,Drain cover on floor is covered with loese tiles- Provide permanent durable cover for area,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",low,1.0,fail,3.0,Inspection failed; violations found,2024.0,2024-09
469218,Mala Town,571486,HE_Fail,2024-09-09 15:37:54+00:00,590.005/5-403.12-C,*,Other Liquid Wastes and Rainwater (C),2024-09-09 15:37:54+00,Fail,Long open trench noted in the lower level- Provide more information to the Health Department for use Refer to plumbing department,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",low,1.0,fail,3.0,Inspection failed; violations found,2024.0,2024-09
469219,Mala Town,571486,HE_Fail,2024-09-09 15:37:54+00:00,590.005/5-501.17-C,*,Toilet Room Receptacle Covered (C),2024-09-09 15:37:54+00,Fail,Waste receptacles in all restrooms with no covers- Provide,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",low,1.0,fail,3.0,Inspection failed; violations found,2024.0,2024-09
469220,Mala Town,571486,HE_Fail,2025-06-30 17:11:13+00:00,590.004/4-101.16-C,*,Sponges Use Limitation (C),2025-06-30 17:11:13+00,Fail,Sponges in use at the 3 bay sink- Discontinue,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",low,1.0,fail,3.0,Inspection failed; violations found,2025.0,2025-06
469221,Mala Town,571486,HE_Fail,2025-06-30 17:11:13+00:00,590.004/4-501.114-P,***,Manual and Mechanical Warewashing Equipment Chemical Sanitization-Temperature pH Concentration and Hardness (P),2025-06-30 17:11:13+00,Fail,Chlorine sanitizer is testing at 0ppm Machine was primed and still had a reading of 0ppm chlorine Service technician was called and all equipment will be cleaned/sanitized in the 3 bay sink until corrections are made,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",high,3.0,fail,3.0,Inspection failed; violations found,2025.0,2025-06
469222,Mala Town,571486,HE_Fail,2025-06-30 17:11:13+00:00,590.005/5-203.14-P,***,Backflow Prevention Device When Required (P),2025-06-30 17:11:13+00,Fail,A has been attached to the faucet at the smaller 3 bay sink in the lower level. Faucet does not have a backflow prevention device and the spray hose was laying in the bottom of the sink- Repair Spray arm for the rinse area of the dishwasher is broken and the nozzle is resting in the sink- Repair,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",high,3.0,fail,3.0,Inspection failed; violations found,2025.0,2025-06
469223,Mala Town,571486,HE_Fail,2025-06-30 17:11:13+00:00,590.005/5-205.11-PF,**,Using a Handwashing Sink-Operation and Maintenance (Pf),2025-06-30 17:11:13+00,Fail,Handsink at the entrance of the kitchen is being used as a dump sink for drink- Properly label and discontinue using as a dump sink,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",medium,2.0,fail,3.0,Inspection failed; violations found,2025.0,2025-06
469224,Mala Town,571486,HE_Fail,2025-11-07 16:49:51+00:00,590.002/2-501.11-PF,**,Clean-up of Vomiting and Diarrheal Events (Pf),2025-11-07 16:49:51+00,Fail,Per person in charge there is no procedures on site for handling a vomit or diarrhea event; provide.,190 HARVARD AV,2134,"(42.35082933643827, -71.1308933256637)",medium,2.0,fail,3.0,Inspection failed; violations found,2025.0,2025-11
469225,Mala 

Test SQL results

In [109]:
# sql 1 test
top_violation_types(2025)

,violdesc,violation_count
0,Nonfood Contact Surfaces (C),1963
1,Controlling Pests (Pf),1873
2,Floors Walls and Ceilings-Cleanability (C),1519
3,Cleaning Ventilation Systems Nuisance and Discharge Prohibition (C),1149
4,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),1121
5,Repairing-Premises Structures Attachments and Fixtures-Methods (C),1097
6,(A)(2) and (B) Time/Temperature Control for Safety Food Hot and Cold Holding (P),1051
7,System Maintained in Good Repair (C),980
8,(A)-(P) Person-In-Charge-Duties (Pf),931
9,Using a Handwashing Sink-Operation and Maintenance (Pf),866


In [61]:
violation_trend_by_month(2025)

,month,violation_count
0,2025-01,3011
1,2025-02,2529
2,2025-03,2890
3,2025-04,3642
4,2025-05,3385
5,2025-06,2818
6,2025-07,3067
7,2025-08,3689
8,2025-09,3030
9,2025-10,5095


In [62]:
violation_trend_by_month_and_severity(2025)

,month,severity_level,violation_count
0,2025-01,high,346
1,2025-01,low,1804
2,2025-01,medium,861
3,2025-02,high,280
4,2025-02,low,1565
5,2025-02,medium,684
6,2025-03,high,262
7,2025-03,low,1777
8,2025-03,medium,851
9,2025-04,high,338


In [64]:
severity_by_zip()

,zip,total_violations,avg_severity,high_severity_count
0,2116.0,37399,1.501110,6608.0
1,2128.0,28663,1.456791,4852.0
2,2115.0,27562,1.464335,4491.0
3,2111.0,24712,1.488427,4138.0
4,2128,21788,1.458601,3747.0
5,2115,20924,1.485471,3742.0
6,2134.0,22530,1.472126,3510.0
7,2130.0,25873,1.382020,3385.0
8,2116,19242,1.487008,3270.0
9,2108.0,17350,1.471412,2912.0


In [169]:
top_violation_owner(2024,40)

,businessname,address,total_violations,serious_outcome_count,avg_severity
0,BOS' Sichuan Taste,204 HARVARD AV,145,139.0,1.503448
1,FIG'S,42 CHARLES ST,118,100.0,1.686441
2,Halal Indian Cuisine,736 HUNTINGTON AV,124,96.0,1.491935
3,Sweet Rice,695 CENTRE ST,123,87.0,1.406504
4,Fritay Restaurant,532 RIVER ST,88,85.0,1.761364
5,Yamato II,545 BOYLSTON ST,97,84.0,1.783505
6,Vegas Restaurant,1592 BLUE HILL AV,90,82.0,1.544444
7,JP Kitchen,3510 WASHINGTON ST,103,80.0,1.592233
8,CAPPY'S PIZZA & SUBS,82 WESTLAND AV,101,79.0,1.346535
9,Red Line Pizza and Grill,582 DORCHESTER AV,92,78.0,1.597826


In [175]:
def low_violation_owner(year=2025, limit=15):
    query = """
    SELECT
        businessname,
        address,
        COUNT(*) AS total_violations,
        SUM(CASE WHEN result_group IN (
            'fail',
            'extended_fail',
            'hearing',
            'temporary_suspension',
            'voluntary_closure_avoid',
            'forced_closure'
        ) THEN 1 ELSE 0 END) AS serious_outcome_count,
        AVG(severity_score) AS avg_severity
    FROM inspections
    WHERE businessname IS NOT NULL
      AND severity_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
    GROUP BY businessname, address
    HAVING COUNT(*) > 10
    ORDER BY serious_outcome_count, avg_severity
    LIMIT ?
    """
    return con.execute(query, [year, year, limit]).df()

In [176]:
low_violation_owner(2024,40)

,businessname,address,total_violations,serious_outcome_count,avg_severity
0,Massiminos Cucina,207 ENDICOTT ST,11,0.0,1.000000
1,TASCA RESTAURANT,1612 COMMONWEALTH AV,12,3.0,1.500000
2,WHOLE FOODS MARKET,15 WASHINGTON ST,11,4.0,1.000000
3,Bunker Hill Knights of Columbus No. 62,545 MEDFORD ST,11,4.0,1.000000
4,Papi's Market,3145 WASHINGTON ST,12,4.0,1.000000
5,Museum of Fine Arts New American Cafe,465 HUNTINGTON AV,11,4.0,1.000000
6,Mcdonalds,540 COMMONWEALTH AV,13,4.0,1.307692
7,Pete's Dockside,12 CHANNEL ST,14,4.0,1.357143
8,Via Cannuccia,1739 DORCHESTER AV,12,4.0,1.500000
9,JENNY'S PIZZA,320 MEDFORD ST,14,4.0,1.500000


Chart function

In [80]:
import plotly.express as px
import statsmodels.api as sm

# create reusable line chart function.
def plot_line(df,x,y,title,color=None):
    fig = px.line(df,x=x,y=y,color=color,title=title,markers=True)

    fig.update_layout(
        template="plotly_white",
        height=550,
        title_x=0.5
    )

    fig.show()

In [84]:
def chart_monthly_violation_trend(df):
    df = df.copy()
    df["month"] = pd.to_datetime(df["month"])

    fig = px.scatter(
        df,
        x="month",
        y="violation_count",
        trendline="ols",
        title="Food Safety Violations Trend"
    )

    fig.add_scatter(
        x=df["month"],
        y=df["violation_count"],
        mode="lines+markers",
        name="Monthly Count",
        #line=dict(color="blue", width=3),
        #marker=dict(color="blue")
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Month",
        yaxis_title="Violation Count"
    )

    fig.show()

In [ ]:
df_trend = violation_trend_by_month()
chart_monthly_violation_trend(df_trend)

Test chart function

In [ ]:
df_trend = violation_trend_by_month()
plot_line(
    df_trend,
    x="month",
    y="violation_count",
    title="Monthly Food Safety RecordedViolations"
)

In [ ]:
def chart_violation_trend_by_severity(df):
    df = df.copy()

    colors = {
        "low": "green",
        "medium": "blue",
        "high": "red"
    }

    fig = px.line(
        df,
        x="month",
        y="violation_count",
        color="severity_level",
        color_discrete_map=colors,
        markers=True,
        title="Monthly Food Safety Recorded Violations by Severity Trend"
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Month",
        yaxis_title="Violation Count"
    )

    fig.show()

In [89]:
df_trend_severity = violation_trend_by_month_and_severity()
chart_violation_trend_by_severity(df_trend_severity)

In [74]:
df_trend_severity = violation_trend_by_month_and_severity()

plot_line(
    df_trend_severity,
    x="month",
    y="violation_count",
    color="severity_level",
    title="Violation Trend by Severity"
)

In [90]:
def chart_top_violation_types(df):
    fig = px.bar(
        df.sort_values("violation_count"),
        x="violation_count",
        y="violdesc",
        orientation="h",
        title="Top Food Inspection Violation Types"
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Violation Count",
        yaxis_title="Violation Type"
    )

    fig.show()

In [110]:
df_top_types = top_violation_types(2025)
chart_top_violation_types(df_top_types)

In [116]:
def chart_top_violation_owner(df, year=None):
    df = df.copy()

    title_year = f" in {year}" if year else " Overall"
    fig = px.bar(
        df.sort_values("serious_outcome_count"),
        x="serious_outcome_count",
        y="businessname",
        orientation="h",
        title=f"Top violation business{title_year}",
        hover_data={
            "total_violations": True,
            "avg_severity": ":.2f",
            "businessname": False
        }
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Serious Outcome Count",
        yaxis_title="Business"
    )

    fig.show()

In [117]:
df_owner = top_violation_owner(year=2024, limit=15)
chart_top_violation_owner(df_owner, year=2024)

Aggregated data would usually surface interesting insights, we would like have understanding high level around questions below. 

•	Which neighborhoods have highest violation rates? 
•	Which cuisines fail most often? 
•	Which violations are increasing? 
•	Which restaurants repeatedly fail? 
•	Which months show spikes? 


6. Deep learning / embeddings

DataFrame row
→ formatted text document
→ embedding vector
→ FAISS index
→ retrieve similar rows
→ LLM summary

Prepare document

we cannot just dump the csv file and chuck it by row as it will lose the header each rows, which is needed. 

In [120]:
data.head(2)

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription_x,inferreddescription_y,year,month
0,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,13-2-304/402.11,*,Clean Cloths Hair Restraint,2018-03-20 14:54:25+00,Fail,One staff person without hair restraint. Provide,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",low,1.0,fail,3.0,Inspection failed; violations found,Inspection failed; violations found,2018.0,2018-03
1,1000 Degrees Pizza,313440,HE_Fail,2018-03-20 14:54:25+00:00,22-4-601/602.11,**,Food Contact Surfaces Clean,2018-03-20 14:54:25+00,Fail,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,02108,"(42.35925954972639, -71.05890048027378)",medium,2.0,fail,3.0,Inspection failed; violations found,Inspection failed; violations found,2018.0,2018-03


In [177]:
#prepare document for each row, only extract meaningful fields 
def build_rag_text(row):
    fields = {
        "business": row.get("businessname"),
        "address": row.get("address"),
        "zip": row.get("zip"),
        "violation": row.get("violdesc"),
        "severity": row.get("severity_level"),
        "inspection result": row.get("result"),
        "inspection result meaning": row.get("inferreddescription"),
        "violation_status": row.get("viol_status"),
        "Inspector Observation / Corrective Action": row.get("comments")
    }
    # join the fields into a single text
    parts = []
    for label, value in fields.items():
        if pd.notna(value) and str(value).strip() != "":
            parts.append(f"{label}: {value}")

    return "\n".join(parts)

In [159]:
# Create rag dataframe from cleaned data
rag_df = data.copy()
# sample to reduce embedding time
rag_df = rag_df.sample(n=50000, random_state=42).reset_index(drop=True)

# Build readable RAG text
rag_df["rag_text"] = rag_df.apply(build_rag_text, axis=1)

In [160]:
#inspect the first string in the rag text field
print(rag_df["rag_text"].iloc[0])

business: Birdies Hot Chicken
address: 245  WASHINGTON ST
zip: 2108.0
violation: Outer Openings  Protected (C)
severity: low
result: HE_Fail
result_meaning: Inspection failed; violations found
violation_status: Fail
comment: Openings on front doors. Repair and provide door sweeps to prevent all day light


Generate embedding

consideration: choose the embedding model that balance cost and efficiency and performance
Experiment with lightweight model all-MiniLM-L6-v2, fast, resource-efficient embeddings.  With just 22M parameters, it delivers solid performance on general semantic search tasks and is used across many production-grade apps.

will try high performance model associated with cost later if the resource is allowed: text-embedding-3-small

In [161]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = rag_df["rag_text"].tolist()

embeddings = embed_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

Build FAISS index

In [162]:
import faiss
import numpy as np

embeddings = embeddings.astype("float32")
print("embeddings shape", embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS index size:", index.ntotal)

embeddings shape (50000, 384)
FAISS index size: 50000


 RAG retrieval

In [163]:
def retrieve_similar_violations(query, k=3):
    #convert query to same embedding
    query_embedding = embed_model.encode(
        #embeds query to a list as faiss expect 2D array
        [query],
        normalize_embeddings=True
    ).astype("float32")
    #search for top k similar violations and store scores and indices
    scores, indices = index.search(query_embedding, k)
    #get the results
    results = rag_df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]

    return results[
        [
            "similarity_score",
            "businessname",
            "address",
            "violdesc",
            "severity_level",
            "result",
            "inferreddescription",
            "comments"
        ]
    ]

In [166]:
data.tail(6)

,businessname,licenseno,result,resultdttm,violation,viol_level,violdesc,violdttm,viol_status,comments,address,zip,location,severity_level,severity_score,result_group,risk_score,inferreddescription,year,month
884600,Zurito/Willie's,574442,HE_Fail,2025-10-16 00:12:49+00:00,590.004/4-303.11-PF,**,Cleaning Agents and Sanitizers Availability (Pf),2025-10-16 00:12:49+00,Fail,At the low temperature dish machine in the kitchen there is no sanitizer bottle; the person in charge went to the storage area and retrieved a bottle of sanitizer but the machine is not pulling sanitizer; sanitizer must always be present for properly sanitizing food contact surfaces. The kitchen staff was using the three compartment sink to wash rinse and sanitize dishes until the warewashing machine in the kitchen is fixed; a service call was placed during the inspection.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",medium,2.0,fail,3.0,Inspection failed; violations found,2025.0,2025-10
884601,Zurito/Willie's,574442,HE_FailExt,2026-03-03 16:27:48+00:00,590.003/3-305.12-C,*,Food Storage Prohibited Areas (C),2026-03-03 16:27:48+00,Fail,Prep Room- All the piping that is exposed needs to be enclosed (sprinkler pipes water pipes electrical lines). Refrigeration room- Pipes need to be need to rewraped and explain what the pipes are used for. Seal the hole in the window where the pipes are located. Hallway- Provide an enclosed ceiling over the ice machine.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",low,1.0,extended_fail,3.0,Extended failure from prior inspection,2026.0,2026-03
884602,Zurito/Willie's,574442,HE_FailExt,2026-03-03 16:27:48+00:00,590.003/3-307.11-C,*,Miscellaneous Sources of Contamination (C),2026-03-03 16:27:48+00,Fail,Mop sink is located next to the prep sink in the basement. Enclose the mop sink.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",low,1.0,extended_fail,3.0,Extended failure from prior inspection,2026.0,2026-03
884603,Zurito/Willie's,574442,HE_FailExt,2026-03-03 16:27:48+00:00,590.006/6-202.15-C,*,Outer Openings Protected (C),2026-03-03 16:27:48+00,Fail,Front exterior door for Willie's needs to be repaired to ensure no rodents can enter.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",low,1.0,extended_fail,3.0,Extended failure from prior inspection,2026.0,2026-03
884606,Zurito/Willie's,574442,HE_Pass,2025-10-23 18:03:50+00:00,590.002/2-402.11-C,*,Effectiveness-Hair Restraints (C),2025-10-23 18:03:50+00,Pass,In the kitchen observed multiple food workers without hair restraints; provide effective hair restraints for all food service workers. The person in charge retrieved hair restraints for the food service workers during the inspection.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",low,1.0,pass,1.0,Inspection passed; no issues found,2025.0,2025-10
884607,Zurito/Willie's,574442,HE_Pass,2025-10-23 18:03:50+00:00,590.004/4-303.11-PF,**,Cleaning Agents and Sanitizers Availability (Pf),2025-10-23 18:03:50+00,Pass,At the low temperature dish machine in the kitchen there is no sanitizer bottle; the person in charge went to the storage area and retrieved a bottle of sanitizer but the machine is not pulling sanitizer; sanitizer must always be present for properly sanitizing food contact surfaces. The kitchen staff was using the three compartment sink to wash rinse and sanitize dishes until the warewashing machine in the kitchen is fixed; a service call was placed during the inspection.,26 CHARLES ST,2108.0,"(42.35674517872961, -71.0699880520806)",medium,2.0,pass,1.0,Inspection passed; no issues found,2025.0,2025-10


In [179]:
query = "Find cases where food quality and safety are the issue "
retrieved_df = retrieve_similar_violations(query,5)
retrieved_df

,similarity_score,businessname,address,violdesc,severity_level,result,inferreddescription,comments
29955,0.559800,The Tavern At The End Of The World,108 CAMBRIDGE ST,Consumption of Animal Foods that are Raw Undercooked or Not Otherwise Processed to Eliminate Pathogens (Pf),medium,HE_Fail,Inspection failed; violations found,New menu items offered under cooked are missing the proper consumer advisory =- address in house
28889,0.556153,UGIS SUBS,62 WARREN ST,Consumer Advisories,high,HE_Pass,Inspection passed; no issues found,POST a consumer advisory on menu board-to read=HAMBURGERS & CHEESEBURGERS ARE COOKED TO ORDER TO YOUR SPECIFICATIONS-CONSUMING RAW OR UNDERCOOKED MEATS POULTRY.SEAFOOD.SHELLFISH OR EGGS MAY INCREASE YOUR RISK OF FOODBORNE ILLNESS-
41070,0.549837,MY DINER,455 E FIRST ST,Consumption of Animal Foods that are Raw Undercooked or Not Otherwise Processed to Eliminate Pathogens (Pf),medium,HE_Fail,Inspection failed; violations found,Disclosure / reminder is incomplete ensure all foods offered under cooked are identified correctly as discussed
39872,0.546399,WORLD'S BEST FOOD MARKET,645 RIVER ST,PIC Knowledge,high,HE_Pass,Inspection passed; no issues found,All food prepartion require a full time or two part time employee how are certified in food safety. Provide
8564,0.543662,IDEAL SUB SHOP,522 DUDLEY ST,Separation Segregation Cross Contamination,high,HE_Fail,Inspection failed; violations found,walk in/properly store all raw meats below ready to eat


RAG summary chain
SQL summary chain
Router chain
Final ask() demo function

Create a small evaluation table with 10–15 test questions.

Columns:

Question
Expected Tool
Actual Tool
Relevant Retrieval? 1/0
Summary Accurate? 1/0
Chart Generated? 1/0
Notes

bench mark vs llm without rag, sql, chart.
Good comparison questions

Use 3–5 examples:

“What are the most common Boston food inspection violations in 2024?”
“Which establishments had the most serious outcomes in 2024?”
“Find violations similar to refrigeration or cold food storage problems.”
“What corrective actions are common for sanitizer machine issues?”
“Are high-severity violations increasing?”
Evaluation table
Question	LLM-only result	RAG/SQL result	Which is better?	Why

8. Router

Chain 1: router

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser

router_template = """
You are a routing assistant for a Boston food safety analytics system.

Choose exactly one tool:

top_violation_types
- Use for questions about the most common violation types.

violation_trend_by_month
- Use for questions about violation trends over time.

violation_trend_by_severity
- Use for questions about trends split by severity.

top_violation_owner
- Use for questions about repeat offenders or businesses with many serious outcomes.

similar_violations
- Use for semantic search questions asking for similar violations, related issues, or examples.

User question:
{text}

Return only the tool name.
"""

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["text"]
)

router_chain = router_prompt | llm | StrOutputParser()

In [ ]:
tool_name = router_chain.invoke({"text": "Which businesses had the most serious violations in 2024?"})
print(tool_name)

In [ ]:
if "top_violation_owner" in tool_name:
    df = top_violation_owner(year=2024)

Chain 2: summary

9. LLM summary

10. Final demo function

In [ ]:
def ask(question):
    route = route_question(question)

    if route == "top":
        df = top_violation_types()
        plot_bar(...)
        return df

    elif route == "trend":
        df = violation_trend()
        plot_line(...)
        return df

    elif route == "semantic":
        return retrieve_similar_violations(question)

Demo:

ask("What are the most common violations?")
ask("Show violation trend over time")
ask("Find violations similar to refrigeration problems")

Reference

https://data.boston.gov/dataset/food-establishment-inspections
https://supermemory.ai/blog/best-open-source-embedding-models-benchmarked-and-ranked/